<a href="https://colab.research.google.com/github/ShaojieDong503/HAD5015-Project/blob/main/Phase_2_StationData_Met_hum.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
stations_path = "/content/drive/MyDrive/ML project data/stations.csv"
stations = pd.read_csv(stations_path)

In [ ]:
stations.head()

,naps_id,lat,lon
0,60104,45.43433,-75.67600
1,60106,45.38287,-75.71387
2,60204,42.31578,-83.04367
3,60211,42.29289,-83.07314
4,60303,44.22008,-76.52141


In [ ]:
import numpy as np
import pandas as pd
from scipy.spatial import cKDTree
from pyproj import Transformer

# Assumes:
# - files_ws: sorted list of parquet paths
# - stations: DataFrame with columns: naps_id, lat, lon

# =========================
# 1) stations -> stations_xy (EPSG:3347)
# =========================
stations = stations[["naps_id", "lat", "lon"]].dropna().drop_duplicates("naps_id").reset_index(drop=True)

transformer = Transformer.from_crs("EPSG:4326", "EPSG:3347", always_xy=True)
x, y = transformer.transform(stations["lon"].to_numpy(), stations["lat"].to_numpy())

stations_xy = stations.copy()
stations_xy["x_3347"] = x
stations_xy["y_3347"] = y

# Optional distance filter (meters)
MAX_DIST_M = None  # e.g., 4000; None disables

In [ ]:
#wind speed
path = "/content/drive/MyDrive/ML project data/DATA/ontario_rhum2m_monthly/ontario_rh2m_2023-12_res5000m.parquet"
wind_speed =  pd.read_parquet(path)
wind_speed.head()

,month,x_3347,y_3347,rh2m
0,2023-12,6.983692e+06,660643.39981,83.459709
1,2023-12,6.988692e+06,660643.39981,83.459709
2,2023-12,6.993692e+06,660643.39981,83.459709
3,2023-12,6.998692e+06,660643.39981,83.496758
4,2023-12,7.003692e+06,660643.39981,83.496758


In [ ]:
import os, glob

def extract_rh2m_station_monthly_wide(
    folder_path: str,
    stations_xy: pd.DataFrame,
    out_csv_path: str,
    pattern: str = "*.parquet",
    month_col: str = "month",
    x_col: str = "x_3347",
    y_col: str = "y_3347",
    value_col: str = "rh2m",
    max_dist_m: float | None = None,
    keep_xy_in_output: bool = False,
):
    """
    Read monthly rh2m parquet grids from folder_path, extract nearest-grid rh2m at station
    (x_3347,y_3347), and save one wide CSV:
      naps_id, lat, lon (+ optional x/y), YYYY-MM columns

    stations_xy must contain: naps_id, lat, lon, x_3347, y_3347 (created externally).
    """

    # ---- checks ----
    req_station = {"naps_id", "lat", "lon", x_col, y_col}
    missing_station = req_station - set(stations_xy.columns)
    if missing_station:
        raise ValueError(f"stations_xy missing columns: {missing_station}")

    files = sorted(glob.glob(os.path.join(folder_path, pattern)))
    if not files:
        raise FileNotFoundError(f"No files matched: {os.path.join(folder_path, pattern)}")

    stations_xy = stations_xy.dropna(subset=["naps_id", "lat", "lon", x_col, y_col]).copy()
    stations_xy = stations_xy.drop_duplicates("naps_id").reset_index(drop=True)

    st_xy = stations_xy[[x_col, y_col]].to_numpy()

    month_to_vals = {}

    for file in files:
        df = pd.read_parquet(file)

        needed = {month_col, x_col, y_col, value_col}
        missing = needed - set(df.columns)
        if missing:
            raise ValueError(f"{os.path.basename(file)} missing columns: {missing}. Found: {list(df.columns)}")

        months = df[month_col].dropna().astype(str).unique()
        for m in months:
            sub = (
                df[df[month_col].astype(str) == m][[x_col, y_col, value_col]]
                .dropna()
                .reset_index(drop=True)
            )

            if sub.empty:
                month_to_vals[m] = np.full(len(stations_xy), np.nan)
                continue

            tree = cKDTree(sub[[x_col, y_col]].to_numpy())
            dist, idx = tree.query(st_xy, k=1)

            vals = sub.loc[idx, value_col].to_numpy()

            if max_dist_m is not None:
                vals = np.where(dist <= max_dist_m, vals, np.nan)

            month_to_vals[m] = vals

    months_sorted = sorted(month_to_vals.keys())
    month_df = pd.DataFrame({m: month_to_vals[m] for m in months_sorted})

    base_cols = ["naps_id", "lat", "lon"]
    if keep_xy_in_output:
        base_cols += [x_col, y_col]

    final = pd.concat([stations_xy[base_cols].reset_index(drop=True), month_df], axis=1)

    # ---- save ----
    os.makedirs(os.path.dirname(out_csv_path), exist_ok=True)
    final.to_csv(out_csv_path, index=False)

    print("Files read:", len(files))
    print("Month columns:", len(months_sorted))
    print("Saved:", out_csv_path)
    print("Shape:", final.shape)

    return final

In [ ]:
folder = "/content/drive/MyDrive/ML project data/DATA/ontario_rhum2m_monthly"
out_csv = "/content/drive/MyDrive/ML project data/rh2m_station_monthly_wide.csv"

rh2m_wide = extract_rh2m_station_monthly_wide(
    folder_path=folder,
    stations_xy=stations_xy,   # created externally
    out_csv_path=out_csv,
    max_dist_m=4000,           # optional
    keep_xy_in_output=False
)

Files read: 168
Month columns: 168
Saved: /content/drive/MyDrive/ML project data/rh2m_station_monthly_wide.csv
Shape: (34, 171)
